In [1]:
# %% [markdown]
# # DIAGNOSTIC: was results_phase3_v2_arch.csv generated with the
# sanitizer active?
#
# Checks the concept_sql the LLM actually generated for the 7
# code-dependent questions (CR19, CR30, CR39, CR40, CR41, CR43, CR59).
# If the sanitizer was active during that run, the LLM should have
# used the new label concepts (operation_label = 'card_withdrawal',
# type_label = 'deposit', etc.) since that's the ONLY way it could
# have known what to filter on -- the raw codes would never have
# appeared anywhere in its prompt. If the sanitizer was NOT active,
# the LLM's concept_sql will show it using the RAW codes directly
# (operation = 'VYBER KARTOU'), because the un-sanitized evidence
# would have handed it those codes verbatim.

# %%
import pandas as pd
pd.set_option('display.max_colwidth', 300)

df = pd.read_csv('results_phase3_v2_arch.csv')

CODE_DEPENDENT_QUESTIONS = ['CR19', 'CR30', 'CR39', 'CR40', 'CR41', 'CR43', 'CR59']
RAW_CODES = ['VYBER KARTOU', 'VYBER', 'PRIJEM', 'VYDAJ', 'VKLAD', 'PREVOD Z UCTU']
LABEL_CONCEPTS = ['operation_label', 'type_label']

print("=" * 70)
print("CHECKING concept_sql FOR RAW CODE vs LABEL CONCEPT USAGE")
print("=" * 70)

sanitizer_was_active_evidence = []
sanitizer_was_inactive_evidence = []

for qid in CODE_DEPENDENT_QUESTIONS:
    sub = df[df['question_id'] == qid]
    if len(sub) == 0:
        print(f"\n[{qid}] NOT FOUND in this results file")
        continue

    print(f"\n[{qid}] category={sub['category'].iloc[0]}")
    for _, row in sub.iterrows():
        concept_sql = str(row.get('concept_sql', ''))
        if concept_sql == 'nan' or not concept_sql:
            continue

        uses_raw_code = any(code in concept_sql for code in RAW_CODES)
        uses_label = any(label in concept_sql for label in LABEL_CONCEPTS)

        if uses_raw_code:
            sanitizer_was_inactive_evidence.append((qid, row['iteration'], concept_sql))
        if uses_label:
            sanitizer_was_active_evidence.append((qid, row['iteration'], concept_sql))

    # show one representative concept_sql for this question
    first_row = sub.iloc[0]
    print(f"  Sample concept_sql (iter{first_row['iteration']}):")
    print(f"    {str(first_row.get('concept_sql', 'N/A'))[:250]}")

print("\n" + "=" * 70)
print("VERDICT")
print("=" * 70)
print(f"Iterations where concept_sql contains a RAW CODE (e.g. 'VYBER KARTOU'): "
      f"{len(sanitizer_was_inactive_evidence)}")
print(f"Iterations where concept_sql contains a LABEL CONCEPT (operation_label/type_label): "
      f"{len(sanitizer_was_active_evidence)}")

if sanitizer_was_inactive_evidence:
    print(f"\n!! FOUND {len(sanitizer_was_inactive_evidence)} case(s) where the LLM used a RAW")
    print("   CODE directly. This means the sanitizer was NOT active during")
    print("   this run -- the un-sanitized evidence handed the LLM the raw")
    print("   code, which it then used directly instead of the label concept.")
    print("\n   Examples:")
    for qid, it, sql in sanitizer_was_inactive_evidence[:5]:
        print(f"\n   [{qid} iter{it}]: {sql[:200]}")
    print("\n   CONCLUSION: results_phase3_v2_arch.csv needs to be regenerated")
    print("   with the sanitizer confirmed active (restart kernel, re-run")
    print("   04_full_experiment.ipynb from a clean import).")
elif sanitizer_was_active_evidence:
    print(f"\n✓ The LLM used LABEL CONCEPTS (operation_label/type_label) and NO")
    print("  raw codes appear anywhere in these questions' concept_sql. This")
    print("  confirms the sanitizer WAS active during this run -- the LLM")
    print("  could only have known to use these label concepts if the raw")
    print("  codes were never in its prompt to begin with.")
    print("\n  CONCLUSION: results_phase3_v2_arch.csv (33.0% accuracy) is a")
    print("  valid, sanitizer-active result and does not need to be rerun")
    print("  for this reason.")
else:
    print("\n?? Neither raw codes nor label concepts found in any concept_sql")
    print("   for these questions -- inspect the sample concept_sql shown")
    print("   above manually; the LLM may have used a different query shape")
    print("   entirely (e.g. avoided filtering on operation/type at all).")

CHECKING concept_sql FOR RAW CODE vs LABEL CONCEPT USAGE

[CR19] category=card_risk
  Sample concept_sql (iter0):
    SELECT AVG(trans.count) 
   FROM trans 
   JOIN account ON trans.account_link 
   JOIN loan ON loan.account_link 
   WHERE loan.status = 'D' AND trans.operation_label = 'card_withdrawal' 
   GROUP BY strftime('%Y-%m', trans.trans_date) 
   HAVING COU

[CR30] category=loan_portfolio
  Sample concept_sql (iter0):
    SELECT 
    account.account_id,
    loan.total_amount / trans.total_amount AS loan_to_deposit_ratio
FROM 
    account
JOIN 
    loan ON loan.account_link
JOIN 
    trans ON trans.account_link
WHERE 
    trans.type_label = 'deposit'
GROUP BY 
    acco

[CR39] category=transaction_behavior
  Sample concept_sql (iter0):
    SELECT SUM(trans.count)
   FROM trans
   WHERE trans.operation_label = 'cash_withdrawal'

[CR40] category=transaction_behavior
  Sample concept_sql (iter0):
    SELECT trans.avg_amount
FROM trans
WHERE trans.operation_label = 'card_withdrawal